In [1]:
!pip -q install transformers sentencepiece evaluate rouge_score

  Installing build dependencies ... done
  Getting requirements to build wheel ... done
  Preparing metadata (pyproject.toml) ... done


In [2]:
import time
import torch
import pandas as pd
import evaluate
from transformers import AutoTokenizer, AutoModelForSeq2SeqLM

device = "cuda" if torch.cuda.is_available() else "cpu"
print("Device:", device)

Device: cuda


In [3]:
article = """Artificial intelligence is changing the way hospitals work around the world. Doctors now use machine learning
models to read X-ray and MRI images, and these systems can detect diseases such as cancer and pneumonia in
very early stages. Hospitals also use AI to predict which patients are likely to need intensive care, so that
beds and staff can be arranged in advance. Chatbots answer simple questions of patients, book appointments and
send reminders for medicines, which reduces the workload of nurses. However, experts warn that these systems
must be trained on diverse and high quality data, otherwise they may give wrong results for some groups of
people. Privacy of patient records is another big concern, and governments are preparing strict rules for the
use of medical data. Most researchers agree that AI will not replace doctors but will help them to take faster
and better decisions."""

reference = ("AI helps hospitals detect diseases early, predict patient needs and reduce staff workload, "
             "but needs good data and privacy rules, and will assist rather than replace doctors.")
print("Article words:", len(article.split()))

Article words: 146


In [4]:
def summarize(model_name, text, prefix="", **gen_kwargs):
    tok = AutoTokenizer.from_pretrained(model_name)
    model = AutoModelForSeq2SeqLM.from_pretrained(model_name).to(device).eval()
    enc = tok(prefix + text, return_tensors="pt", truncation=True, max_length=1024).to(device)
    start = time.time()
    with torch.no_grad():
        out = model.generate(**enc, **gen_kwargs)
    return tok.decode(out[0], skip_special_tokens=True), round(time.time() - start, 2)

gen = dict(num_beams=4, max_length=70, min_length=25, early_stopping=True, no_repeat_ngram_size=3)

bart_sum, bart_t = summarize("facebook/bart-large-cnn", article, **gen)
print(f"BART summary ({bart_t}s):\n{bart_sum}\n")

config.json:   0%|          | 0.00/1.58k [00:00<?, ?B/s]

vocab.json:   0%|          | 0.00/899k [00:00<?, ?B/s]

merges.txt:   0%|          | 0.00/456k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.36M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B / 1.63GB            

model.safetensors: downloading bytes:           |  0.00B            

[transformers] Please make sure the generation config includes `forced_bos_token_id=0`. 


Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/363 [00:00<?, ?B/s]

BART summary (3.27s):
Artificial intelligence is changing the way hospitals work around the world. Doctors now use machine learning models to read X-ray and MRI images. Hospitals also use AI to predict which patients are likely to need intensive care.



In [5]:
t5_sum, t5_t = summarize("t5-small", article, prefix="summarize: ", **gen)
print(f"T5-small summary ({t5_t}s):\n{t5_sum}")

config.json:   0%|          | 0.00/1.21k [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/2.32k [00:00<?, ?B/s]

spiece.model:   0%|          | 0.00/792k [00:00<?, ?B/s]

tokenizer.json:   0%|          | 0.00/1.39M [00:00<?, ?B/s]

model.safetensors: reconstructing file:   0%|          |  0.00B /  242MB            

model.safetensors: downloading bytes:           |  0.00B            

Loading weights:   0%|          | 0/131 [00:00<?, ?it/s]

generation_config.json:   0%|          | 0.00/147 [00:00<?, ?B/s]

T5-small summary (1.52s):
doctors now use machine learning models to read X-ray and MRI images. these systems can detect diseases such as cancer and pneumonia in early stages. hospitals also use AI to predict which patients are likely to need intensive care.


In [6]:
# ROUGE evaluation against the reference summary
rouge = evaluate.load("rouge")
rows = []
for name, s in [("BART", bart_sum), ("T5-small", t5_sum)]:
    r = rouge.compute(predictions=[s], references=[reference])
    rows.append((name, len(s.split()), round(r["rouge1"], 3), round(r["rouge2"], 3), round(r["rougeL"], 3)))
display(pd.DataFrame(rows, columns=["model", "summary words", "ROUGE-1", "ROUGE-2", "ROUGE-L"]))
print("Compression: article", len(article.split()), "words ->", len(bart_sum.split()), "words (BART)")

,model,summary words,ROUGE-1,ROUGE-2,ROUGE-L
0,BART,37,0.154,0.000,0.062
1,T5-small,39,0.269,0.031,0.119


Compression: article 146 words -> 37 words (BART)


In [7]:
# Effect of summary length (BART)
for mx in [30, 60, 100]:
    s, _ = summarize("facebook/bart-large-cnn", article, num_beams=4, max_length=mx, min_length=10, early_stopping=True)
    print(f"max_length={mx:3d} -> {s}\n")

Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

max_length= 30 -> Artificial intelligence is changing the way hospitals work around the world. Doctors now use machine learning to read X-ray and MRI images.



Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

max_length= 60 -> Artificial intelligence is changing the way hospitals work around the world. Doctors now use machine learning models to read X-ray and MRI images. Hospitals also use AI to predict which patients are likely to need intensive care.



Loading weights:   0%|          | 0/511 [00:00<?, ?it/s]

max_length=100 -> Artificial intelligence is changing the way hospitals work around the world. Doctors now use machine learning models to read X-ray and MRI images. Hospitals also use AI to predict which patients are likely to need intensive care.

